# 01 — CNN Standalone Model

Custom 4-layer CNN trained on 224×224 MRI images for 4-class brain tumor classification.
Saves `cnn_model.h5` to `../saved_models/` for use in 02_CNN_Ensemble.ipynb and 08_AllModelsCombined.ipynb.

## Section 1: Imports & Configuration

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
print("\u2713 Imports complete")

## Section 2: Constants & Hyperparameters

In [ ]:
NOTEBOOK_NAME = "01_CNN_Standalone"

# Dataset
DATASET_PATH     = "../MRI_DATASET/"
TRAIN_DIR        = DATASET_PATH + "Training/"
TEST_DIR         = DATASET_PATH + "Testing/"

# Classes — FIXED ORDER, DO NOT CHANGE
CLASS_NAMES      = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES      = 4

# Image settings
IMG_HEIGHT       = 224
IMG_WIDTH        = 224
CHANNELS         = 3

# Training hyperparameters
BATCH_SIZE       = 32
EPOCHS           = 20
LEARNING_RATE    = 1e-4
VALIDATION_SPLIT = 0.2

# Reproducibility
RANDOM_SEED      = 42

# Paths
SAVED_MODELS_DIR = "../saved_models/"
os.makedirs(SAVED_MODELS_DIR, exist_ok=True)

# Set seeds
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
os.environ['PYTHONHASHSEED'] = str(RANDOM_SEED)

print("\u2713 Constants configured")
print(f"  Image size : {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"  Batch size : {BATCH_SIZE}")
print(f"  Epochs     : {EPOCHS}")
print(f"  Classes    : {CLASS_NAMES}")

## Section 3: Data Loading & Verification

In [ ]:
# --- Augmentation for training data ---
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=VALIDATION_SPLIT
)

# --- No augmentation for test/validation data ---
test_datagen = ImageDataGenerator(rescale=1./255)

# --- Training generator ---
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='training',
    seed=RANDOM_SEED,
    shuffle=True
)

# --- Validation generator ---
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    subset='validation',
    seed=RANDOM_SEED,
    shuffle=False
)

# --- Test generator ---
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=CLASS_NAMES,
    shuffle=False
)

# --- Verification ---
print("=" * 50)
print("DATA VERIFICATION")
print(f"Class indices      : {train_generator.class_indices}")
print(f"Training samples   : {train_generator.samples}")
print(f"Validation samples : {val_generator.samples}")
print(f"Test samples       : {test_generator.samples}")
print(f"Image size         : {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"Batch size         : {BATCH_SIZE}")
print("=" * 50)

## Section 4: Data Preprocessing & Augmentation

In [ ]:
# Augmentation is defined in the ImageDataGenerator in Section 3.
# No additional preprocessing needed — rescale to [0,1] and augment on the fly.
print("\u2713 Data augmentation configured via ImageDataGenerator")

## Section 5: Model Definition

In [ ]:
# --- Custom CNN architecture ---
# 4\u00d7 (Conv2D \u2192 MaxPooling), then Dense feature layer, Dropout, softmax output
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu', name='feature_layer'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax', name='output_layer'),
], name='cnn_model')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print("\u2713 Model defined and compiled")

## Section 6: Model Training

In [ ]:
# --- Callbacks ---
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(
        filepath=SAVED_MODELS_DIR + 'cnn_model_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

# --- Train ---
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\u2713 Training complete")

# --- Plot training curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.suptitle(f'{NOTEBOOK_NAME} \u2014 Training History')
plt.tight_layout()
plt.show()

## Section 7: Model Evaluation

In [ ]:
def evaluate_model(model, generator, model_name="Model"):
    """Standard evaluation: confusion matrix, classification report, ROC, PR curves."""
    generator.reset()
    y_pred_proba = model.predict(generator, verbose=1)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    y_true       = generator.classes

    # --- Confusion Matrix ---
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f'{model_name} \u2014 Confusion Matrix')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.show()

    # --- Classification Report ---
    print(f"\n{model_name} \u2014 Classification Report")
    print("=" * 60)
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES)
    print(report)

    # --- ROC Curve ---
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{cls} (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} \u2014 ROC Curve')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

    # --- Precision-Recall Curve ---
    plt.figure(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        precision, recall, _ = precision_recall_curve(y_true_bin[:, i], y_pred_proba[:, i])
        ap = average_precision_score(y_true_bin[:, i], y_pred_proba[:, i])
        plt.plot(recall, precision, label=f'{cls} (AP = {ap:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'{model_name} \u2014 Precision-Recall Curve')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    return y_pred, y_pred_proba

# --- Evaluate on test set ---
y_pred, y_pred_proba = evaluate_model(model, test_generator, model_name=NOTEBOOK_NAME)
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n\u2713 Test Accuracy : {test_acc:.4f}")
print(f"\u2713 Test Loss     : {test_loss:.4f}")

## Section 8: Save Model

In [ ]:
# --- Save final model ---
model_path = SAVED_MODELS_DIR + 'cnn_model.h5'
model.save(model_path)
print(f"\u2713 Model saved to: {model_path}")
print("  This model is used as:")
print("  1. Standalone CNN classifier (loaded in 08_AllModelsCombined.ipynb)")
print("  2. Feature extractor backbone (loaded in 02_CNN_Ensemble.ipynb)")

## Section 9: Results Summary

In [ ]:
print("=" * 60)
print(f"NOTEBOOK: {NOTEBOOK_NAME}")
print(f"Dataset  : {train_generator.samples + val_generator.samples} training images")
print(f"Classes  : {CLASS_NAMES}")
print(f"Image size: {IMG_HEIGHT}\u00d7{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {EPOCHS}, LR: {LEARNING_RATE}")
print(f"Seed     : {RANDOM_SEED}")
print("-" * 60)
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")
print("=" * 60)
print("Saved models location:", SAVED_MODELS_DIR)